In [ ]:
# Step 1 — Load mutation-window and audited clinical datasets

from pathlib import Path
import pandas as pd
import numpy as np
import json

PROJECT_DIR = Path(
    "C:\Users\LAPTOP\Downloads\luad_reserch_code"
)

MUTATION_WINDOWS_PATH = (
    PROJECT_DIR
    / "luad_processed"
    / "luad_mutation_adjusted_windows_v3.csv"
)

CLINICAL_DATA_PATH = (
    PROJECT_DIR
    / "luad_processeed_data"
    / "luad_improved_clinical_covariates.csv"
)

IMPROVED_OUTPUT_DIR = (
    PROJECT_DIR
    / "luad_processeed_data"
)

print("Mutation-window file exists:", MUTATION_WINDOWS_PATH.exists())
print("Clinical file exists:", CLINICAL_DATA_PATH.exists())
print("Output folder exists:", IMPROVED_OUTPUT_DIR.exists())

mutation_df = pd.read_csv(
    MUTATION_WINDOWS_PATH,
    low_memory=False
)

clinical_df = pd.read_csv(
    CLINICAL_DATA_PATH,
    low_memory=False
)

print("\nMutation dataset shape:", mutation_df.shape)
print("Clinical dataset shape:", clinical_df.shape)

print("\nMutation dataset columns:")
print(mutation_df.columns.tolist())

Mutation-window file exists: True
Clinical file exists: True
Output folder exists: True

Mutation dataset shape: (15142, 42)
Clinical dataset shape: (3382, 15)

Mutation dataset columns:
['COHORT', 'SAMPLE_ID', 'PATIENT_ID', 'SEX', 'sex_encoded', 'STAGE_LABEL_4CLASS', 'STAGE_BINARY', 'label_stage_4class', 'label_stage_binary', 'GENE_SYMBOL', 'ENSEMBL_GENE_ID', 'TRANSCRIPT_ID_RAW', 'TRANSCRIPT_ID_MATCHED', 'TRANSCRIPT_MATCH_METHOD', 'HGVSc_RAW', 'HGVSc_CLEAN', 'HGVSp_Short', 'Variant_Classification', 'Variant_Type', 'Chromosome', 'Start_Position', 'End_Position', 'Reference_Allele', 'Tumor_Seq_Allele1', 'Tumor_Seq_Allele2', 'Protein_position', 'MUTATION_CATEGORY', 'MUTATION_TOKEN', 'REFERENCE_CDS_LENGTH', 'REFERENCE_DNA_WINDOW', 'MUTATED_DNA_WINDOW', 'WINDOW_LENGTH', 'MUTATION_INDEX_IN_WINDOW', 'SEQUENCE_METHOD', 'TRANSCRIPT_MATCHED', 'HGVSC_PARSEABLE', 'REFERENCE_ALLELE_MATCHED', 'MUTATION_APPLIED', 'REFERENCE_WINDOW_VALID', 'MUTATED_WINDOW_VALID', 'FALLBACK_AVAILABLE', 'VALIDATION_NOT

In [2]:
# Step 2 — Audit mutation features before creating masks

required_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "GENE_SYMBOL",
    "Variant_Classification",
    "Variant_Type",
    "Reference_Allele",
    "Tumor_Seq_Allele2",
    "REFERENCE_DNA_WINDOW",
    "MUTATED_DNA_WINDOW",
    "WINDOW_LENGTH",
    "MUTATION_INDEX_IN_WINDOW",
    "MUTATION_APPLIED",
    "REFERENCE_WINDOW_VALID",
    "MUTATED_WINDOW_VALID"
]

missing_columns = [
    column
    for column in required_columns
    if column not in mutation_df.columns
]

print("Missing required columns:", missing_columns)

# Convert numeric fields safely
mutation_df["WINDOW_LENGTH_NUMERIC"] = pd.to_numeric(
    mutation_df["WINDOW_LENGTH"],
    errors="coerce"
)

mutation_df["MUTATION_INDEX_NUMERIC"] = pd.to_numeric(
    mutation_df["MUTATION_INDEX_IN_WINDOW"],
    errors="coerce"
)

print("\nDataset rows:", len(mutation_df))

print("\nValidation flags:")
for column in [
    "MUTATION_APPLIED",
    "REFERENCE_WINDOW_VALID",
    "MUTATED_WINDOW_VALID",
    "REFERENCE_ALLELE_MATCHED",
    "HGVSC_PARSEABLE"
]:
    if column in mutation_df.columns:
        print(f"\n{column}")
        print(
            mutation_df[column]
            .fillna("MISSING")
            .value_counts(dropna=False)
            .to_string()
        )

print("\nWindow-length distribution:")
print(
    mutation_df["WINDOW_LENGTH_NUMERIC"]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

valid_index = (
    mutation_df["MUTATION_INDEX_NUMERIC"].notna()
    & (mutation_df["MUTATION_INDEX_NUMERIC"] >= 0)
    & (
        mutation_df["MUTATION_INDEX_NUMERIC"]
        < mutation_df["WINDOW_LENGTH_NUMERIC"]
    )
)

print("\nMutation-position audit:")
print(
    "Valid mutation indices:",
    int(valid_index.sum()),
    "/",
    len(mutation_df)
)

print(
    "Invalid or missing indices:",
    int((~valid_index).sum())
)

print(
    "Minimum valid index:",
    mutation_df.loc[
        valid_index,
        "MUTATION_INDEX_NUMERIC"
    ].min()
)

print(
    "Maximum valid index:",
    mutation_df.loc[
        valid_index,
        "MUTATION_INDEX_NUMERIC"
    ].max()
)

print("\nVariant classification counts:")
print(
    mutation_df["Variant_Classification"]
    .fillna("MISSING")
    .value_counts()
    .to_string()
)

print("\nExisting mutation-category counts:")
print(
    mutation_df["MUTATION_CATEGORY"]
    .fillna("MISSING")
    .value_counts()
    .to_string()
)

print("\nVariant-type counts:")
print(
    mutation_df["Variant_Type"]
    .fillna("MISSING")
    .value_counts()
    .to_string()
)

print("\nSample mutation records:")
display(
    mutation_df[
        [
            "COHORT",
            "PATIENT_ID",
            "GENE_SYMBOL",
            "Variant_Classification",
            "Variant_Type",
            "Reference_Allele",
            "Tumor_Seq_Allele2",
            "WINDOW_LENGTH",
            "MUTATION_INDEX_IN_WINDOW",
            "MUTATION_CATEGORY",
            "MUTATION_APPLIED"
        ]
    ].head(10)
)

Missing required columns: []

Dataset rows: 15142

Validation flags:

MUTATION_APPLIED
MUTATION_APPLIED
1    12689
0     2453

REFERENCE_WINDOW_VALID
REFERENCE_WINDOW_VALID
1    13106
0     2036

MUTATED_WINDOW_VALID
MUTATED_WINDOW_VALID
1    12689
0     2453

REFERENCE_ALLELE_MATCHED
REFERENCE_ALLELE_MATCHED
1    12689
0     2453

HGVSC_PARSEABLE
HGVSC_PARSEABLE
1    13779
0     1363

Window-length distribution:
WINDOW_LENGTH_NUMERIC
0       2036
101    13106

Mutation-position audit:
Valid mutation indices: 13106 / 15142
Invalid or missing indices: 2036
Minimum valid index: 50.0
Maximum valid index: 50.0

Variant classification counts:
Variant_Classification
Missense_Mutation         9445
Nonsense_Mutation         1624
Frame_Shift_Del            927
Silent                     773
Splice_Site                752
In_Frame_Del               575
Frame_Shift_Ins            296
Intron                     296
In_Frame_Ins               156
Splice_Region              107
3'UTR                

,COHORT,PATIENT_ID,GENE_SYMBOL,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele2,WINDOW_LENGTH,MUTATION_INDEX_IN_WINDOW,MUTATION_CATEGORY,MUTATION_APPLIED
0,TCGA_Atlas_2018,TCGA-05-4244,ATM,Missense_Mutation,SNP,A,C,101,50.0,SNV,1
1,TCGA_Atlas_2018,TCGA-05-4244,KRAS,Missense_Mutation,SNP,C,A,101,50.0,SNV,1
2,TCGA_Atlas_2018,TCGA-05-4244,ANK2,Silent,SNP,C,A,101,50.0,SNV,1
3,TCGA_Atlas_2018,TCGA-05-4244,ANK1,Missense_Mutation,SNP,C,T,101,50.0,SNV,1
4,TCGA_Atlas_2018,TCGA-05-4244,ZFHX4,Nonsense_Mutation,SNP,G,T,0,NaN,SNV,0
5,TCGA_Atlas_2018,TCGA-05-4244,KDM5C,Nonsense_Mutation,SNP,G,A,101,50.0,SNV,1
6,TCGA_Atlas_2018,TCGA-05-4249,KRAS,Missense_Mutation,SNP,C,A,101,50.0,SNV,1
7,TCGA_Atlas_2018,TCGA-05-4249,ARID2,Silent,SNP,G,T,101,50.0,SNV,1
8,TCGA_Atlas_2018,TCGA-05-4249,LRP1B,Splice_Site,SNP,T,A,0,NaN,UNSUPPORTED_HGVSC,0
9,TCGA_Atlas_2018,TCGA-05-4249,PIK3CA,Missense_Mutation,SNP,G,A,101,50.0,SNV,1


In [3]:
# Step 3 — Retain fully valid mutations and inspect allele spans

valid_mutation_df = mutation_df.copy()

# Convert validation flags safely to numeric
validation_columns = [
    "MUTATION_APPLIED",
    "REFERENCE_WINDOW_VALID",
    "MUTATED_WINDOW_VALID",
    "REFERENCE_ALLELE_MATCHED"
]

for column in validation_columns:
    valid_mutation_df[column] = pd.to_numeric(
        valid_mutation_df[column],
        errors="coerce"
    ).fillna(0).astype(int)

valid_mutation_df["WINDOW_LENGTH_NUMERIC"] = pd.to_numeric(
    valid_mutation_df["WINDOW_LENGTH"],
    errors="coerce"
)

valid_mutation_df["MUTATION_INDEX_NUMERIC"] = pd.to_numeric(
    valid_mutation_df["MUTATION_INDEX_IN_WINDOW"],
    errors="coerce"
)

# Keep only fully valid 101-base mutation windows
valid_row_mask = (
    (valid_mutation_df["MUTATION_APPLIED"] == 1)
    & (valid_mutation_df["REFERENCE_WINDOW_VALID"] == 1)
    & (valid_mutation_df["MUTATED_WINDOW_VALID"] == 1)
    & (valid_mutation_df["REFERENCE_ALLELE_MATCHED"] == 1)
    & (valid_mutation_df["WINDOW_LENGTH_NUMERIC"] == 101)
    & valid_mutation_df["MUTATION_INDEX_NUMERIC"].between(
        0,
        100,
        inclusive="both"
    )
)

valid_mutation_df = (
    valid_mutation_df.loc[valid_row_mask]
    .copy()
    .reset_index(drop=True)
)

valid_mutation_df["MUTATION_INDEX"] = (
    valid_mutation_df["MUTATION_INDEX_NUMERIC"]
    .astype(int)
)


# Clean reference and alternate alleles
def clean_allele(value):
    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    if value in ["-", ".", "NA", "NAN", "NONE"]:
        return ""

    return value


valid_mutation_df["REFERENCE_ALLELE_CLEAN"] = (
    valid_mutation_df["Reference_Allele"]
    .apply(clean_allele)
)

valid_mutation_df["ALTERNATE_ALLELE_CLEAN"] = (
    valid_mutation_df["Tumor_Seq_Allele2"]
    .apply(clean_allele)
)

valid_mutation_df["REFERENCE_ALLELE_LENGTH"] = (
    valid_mutation_df["REFERENCE_ALLELE_CLEAN"]
    .str.len()
)

valid_mutation_df["ALTERNATE_ALLELE_LENGTH"] = (
    valid_mutation_df["ALTERNATE_ALLELE_CLEAN"]
    .str.len()
)


# Assign an event type for mask construction
def classify_mask_event(row):
    ref_length = row["REFERENCE_ALLELE_LENGTH"]
    alt_length = row["ALTERNATE_ALLELE_LENGTH"]

    if ref_length == 1 and alt_length == 1:
        return "SNV"

    if ref_length == 0 and alt_length > 0:
        return "INSERTION"

    if ref_length > 0 and alt_length == 0:
        return "DELETION"

    if ref_length == alt_length and ref_length > 1:
        return "MULTI_BASE_SUBSTITUTION"

    return "COMPLEX_REPLACEMENT"


valid_mutation_df["MASK_EVENT_TYPE"] = (
    valid_mutation_df.apply(
        classify_mask_event,
        axis=1
    )
)

# Proposed number of marked positions:
# insertion uses one anchor position
valid_mutation_df["PROPOSED_MASK_SPAN"] = np.where(
    valid_mutation_df["MASK_EVENT_TYPE"] == "INSERTION",
    1,
    valid_mutation_df["REFERENCE_ALLELE_LENGTH"].clip(lower=1)
)

print("Original mutation rows:", len(mutation_df))
print("Fully valid rows retained:", len(valid_mutation_df))
print("Rows excluded:", len(mutation_df) - len(valid_mutation_df))

print("\nMask event types:")
print(
    valid_mutation_df["MASK_EVENT_TYPE"]
    .value_counts()
    .to_string()
)

print("\nVariant classifications:")
print(
    valid_mutation_df["Variant_Classification"]
    .fillna("MISSING")
    .value_counts()
    .to_string()
)

print("\nProposed mask-span distribution:")
print(
    valid_mutation_df["PROPOSED_MASK_SPAN"]
    .value_counts()
    .sort_index()
    .head(20)
    .to_string()
)

print("\nMaximum proposed mask span:")
print(valid_mutation_df["PROPOSED_MASK_SPAN"].max())

print("\nExample non-SNV mutations:")
display(
    valid_mutation_df.loc[
        valid_mutation_df["MASK_EVENT_TYPE"] != "SNV",
        [
            "COHORT",
            "PATIENT_ID",
            "GENE_SYMBOL",
            "Variant_Classification",
            "Variant_Type",
            "Reference_Allele",
            "Tumor_Seq_Allele2",
            "REFERENCE_ALLELE_LENGTH",
            "ALTERNATE_ALLELE_LENGTH",
            "MUTATION_INDEX",
            "MASK_EVENT_TYPE",
            "PROPOSED_MASK_SPAN"
        ]
    ].head(15)
)

Original mutation rows: 15142
Fully valid rows retained: 12689
Rows excluded: 2453

Mask event types:
MASK_EVENT_TYPE
SNV                        10637
DELETION                    1330
INSERTION                    426
COMPLEX_REPLACEMENT          155
MULTI_BASE_SUBSTITUTION      141

Variant classifications:
Variant_Classification
Missense_Mutation         8652
Nonsense_Mutation         1453
Frame_Shift_Del            901
Silent                     652
In_Frame_Del               554
Frame_Shift_Ins            273
In_Frame_Ins               152
Splice_Region               36
Translation_Start_Site      10
Splice_Site                  5
Nonstop_Mutation             1

Proposed mask-span distribution:
PROPOSED_MASK_SPAN
1     11612
2       240
3        92
4        72
5        25
6        24
7        17
8        13
9        18
10       26
11       11
12       11
13       12
14        7
15      327
16       11
17       12
18       65
19       18
20       11

Maximum proposed mask span:
73

E

,COHORT,PATIENT_ID,GENE_SYMBOL,Variant_Classification,Variant_Type,Reference_Allele,Tumor_Seq_Allele2,REFERENCE_ALLELE_LENGTH,ALTERNATE_ALLELE_LENGTH,MUTATION_INDEX,MASK_EVENT_TYPE,PROPOSED_MASK_SPAN
18,TCGA_Atlas_2018,TCGA-05-4382,NBEA,Frame_Shift_Del,DEL,C,-,1,0,50,DELETION,1
64,TCGA_Atlas_2018,TCGA-05-4389,ANK2,Frame_Shift_Del,DEL,A,-,1,0,50,DELETION,1
80,TCGA_Atlas_2018,TCGA-05-4390,KDM5C,Frame_Shift_Del,DEL,C,-,1,0,50,DELETION,1
92,TCGA_Atlas_2018,TCGA-05-4396,ARID2,Frame_Shift_Del,DEL,G,-,1,0,50,DELETION,1
96,TCGA_Atlas_2018,TCGA-05-4396,TP53,Frame_Shift_Del,DEL,T,-,1,0,50,DELETION,1
122,TCGA_Atlas_2018,TCGA-05-4397,TP53,Frame_Shift_Del,DEL,C,-,1,0,50,DELETION,1
150,TCGA_Atlas_2018,TCGA-05-4402,TP53,Frame_Shift_Del,DEL,TGGGGGCAGCGCCTCA,-,16,0,50,DELETION,16
151,TCGA_Atlas_2018,TCGA-05-4402,APC,Frame_Shift_Ins,INS,-,A,0,1,50,INSERTION,1
152,TCGA_Atlas_2018,TCGA-05-4402,EGFR,In_Frame_Del,DEL,CATCTCCGAAAGCCAACAAGGAAAT,A,25,1,50,COMPLEX_REPLACEMENT,25
160,TCGA_Atlas_2018,TCGA-05-4405,MGA,Frame_Shift_Del,DEL,G,-,1,0,50,DELETION,1


In [4]:
# Step 4 — Create mutation-position masks of length 101

WINDOW_SIZE = 101


def get_mask_span(row):
    """
    Return the number of positions to mark in the mutation mask.

    SNV:
        Mark one position.

    Insertion:
        Mark the inserted sequence length.

    Deletion:
        Mark the deleted reference sequence length.

    Substitution/replacement:
        Mark the larger affected allele span.
    """
    event_type = row["MASK_EVENT_TYPE"]

    ref_length = int(row["REFERENCE_ALLELE_LENGTH"])
    alt_length = int(row["ALTERNATE_ALLELE_LENGTH"])

    if event_type == "SNV":
        return 1

    if event_type == "INSERTION":
        return max(alt_length, 1)

    if event_type == "DELETION":
        return max(ref_length, 1)

    if event_type == "MULTI_BASE_SUBSTITUTION":
        return max(ref_length, alt_length, 1)

    if event_type == "COMPLEX_REPLACEMENT":
        return max(ref_length, alt_length, 1)

    return 1


def create_position_mask(row):
    """
    Create a binary mask aligned with the 101-base DNA window.

    MUTATION_INDEX is treated as a zero-based index.
    """
    mask = [0] * WINDOW_SIZE

    mutation_index = int(row["MUTATION_INDEX"])
    requested_span = int(row["MASK_SPAN_REQUESTED"])

    if mutation_index < 0 or mutation_index >= WINDOW_SIZE:
        return mask

    mask_end = min(
        mutation_index + requested_span,
        WINDOW_SIZE
    )

    for position in range(mutation_index, mask_end):
        mask[position] = 1

    return mask


# Calculate full affected span
valid_mutation_df["MASK_SPAN_REQUESTED"] = (
    valid_mutation_df.apply(
        get_mask_span,
        axis=1
    )
)

# Create binary masks
valid_mutation_df["MUTATION_POSITION_MASK"] = (
    valid_mutation_df.apply(
        create_position_mask,
        axis=1
    )
)

# Mask validation fields
valid_mutation_df["MASK_LENGTH"] = (
    valid_mutation_df["MUTATION_POSITION_MASK"]
    .apply(len)
)

valid_mutation_df["MASK_MARKED_POSITIONS"] = (
    valid_mutation_df["MUTATION_POSITION_MASK"]
    .apply(sum)
)

valid_mutation_df["MASK_CLIPPED_FLAG"] = (
    valid_mutation_df["MASK_MARKED_POSITIONS"]
    < valid_mutation_df["MASK_SPAN_REQUESTED"]
).astype(int)

valid_mutation_df["MASK_BINARY_VALID"] = (
    valid_mutation_df["MUTATION_POSITION_MASK"]
    .apply(
        lambda mask: set(mask).issubset({0, 1})
    )
)

valid_mutation_df["MASK_VALID"] = (
    (valid_mutation_df["MASK_LENGTH"] == WINDOW_SIZE)
    & valid_mutation_df["MASK_BINARY_VALID"]
    & (valid_mutation_df["MASK_MARKED_POSITIONS"] >= 1)
)


# Store marked positions for easy inspection
valid_mutation_df["MASKED_POSITION_INDICES"] = (
    valid_mutation_df["MUTATION_POSITION_MASK"]
    .apply(
        lambda mask: [
            index
            for index, value in enumerate(mask)
            if value == 1
        ]
    )
)


print("Total masks created:", len(valid_mutation_df))

print(
    "Valid masks:",
    int(valid_mutation_df["MASK_VALID"].sum())
)

print(
    "Invalid masks:",
    int((~valid_mutation_df["MASK_VALID"]).sum())
)

print(
    "Masks clipped at window boundary:",
    int(valid_mutation_df["MASK_CLIPPED_FLAG"].sum())
)

print("\nMarked-position distribution:")
print(
    valid_mutation_df["MASK_MARKED_POSITIONS"]
    .value_counts()
    .sort_index()
    .head(20)
    .to_string()
)

print("\nMaximum marked positions:")
print(
    valid_mutation_df["MASK_MARKED_POSITIONS"].max()
)

print("\nExample masks by mutation type:")

display(
    valid_mutation_df.groupby(
        "MASK_EVENT_TYPE",
        group_keys=False
    ).head(2)[
        [
            "GENE_SYMBOL",
            "Variant_Classification",
            "Reference_Allele",
            "Tumor_Seq_Allele2",
            "MUTATION_INDEX",
            "MASK_EVENT_TYPE",
            "MASK_SPAN_REQUESTED",
            "MASK_MARKED_POSITIONS",
            "MASKED_POSITION_INDICES",
            "MASK_VALID"
        ]
    ]
)

Total masks created: 12689
Valid masks: 12689
Invalid masks: 0
Masks clipped at window boundary: 2

Marked-position distribution:
MASK_MARKED_POSITIONS
1     11417
2       252
3       122
4        79
5        30
6        41
7        17
8        17
9        57
10       29
11       11
12       70
13       13
14        7
15      328
16       13
17       12
18       74
19       19
20       11

Maximum marked positions:
51

Example masks by mutation type:


,GENE_SYMBOL,Variant_Classification,Reference_Allele,Tumor_Seq_Allele2,MUTATION_INDEX,MASK_EVENT_TYPE,MASK_SPAN_REQUESTED,MASK_MARKED_POSITIONS,MASKED_POSITION_INDICES,MASK_VALID
0,ATM,Missense_Mutation,A,C,50,SNV,1,1,[50],True
1,KRAS,Missense_Mutation,C,A,50,SNV,1,1,[50],True
18,NBEA,Frame_Shift_Del,C,-,50,DELETION,1,1,[50],True
64,ANK2,Frame_Shift_Del,A,-,50,DELETION,1,1,[50],True
151,APC,Frame_Shift_Ins,-,A,50,INSERTION,1,1,[50],True
152,EGFR,In_Frame_Del,CATCTCCGAAAGCCAACAAGGAAAT,A,50,COMPLEX_REPLACEMENT,25,25,"[50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 6...",True
254,SETD2,Frame_Shift_Ins,-,A,50,INSERTION,1,1,[50],True
453,TP53,Frame_Shift_Del,CGG,GC,50,COMPLEX_REPLACEMENT,3,3,"[50, 51, 52]",True
4049,SMARCA4,Missense_Mutation,GG,TT,50,MULTI_BASE_SUBSTITUTION,2,2,"[50, 51]",True
4070,KEAP1,Nonsense_Mutation,CGCCC,AGCCA,50,MULTI_BASE_SUBSTITUTION,5,5,"[50, 51, 52, 53, 54]",True


In [5]:
# Step 5 — Audit the 2,453 excluded mutation rows for systematic bias

mutation_exclusion_audit_df = mutation_df.copy()

# The valid_row_mask was created in Step 3
mutation_exclusion_audit_df["ROW_STATUS"] = np.where(
    valid_row_mask,
    "Retained",
    "Excluded"
)

mutation_exclusion_audit_df["Variant_Classification"] = (
    mutation_exclusion_audit_df["Variant_Classification"]
    .fillna("MISSING")
    .astype(str)
    .str.strip()
)

overall_exclusion_rate = (
    mutation_exclusion_audit_df["ROW_STATUS"]
    .eq("Excluded")
    .mean()
    * 100
)

print("Total mutation rows:", len(mutation_exclusion_audit_df))
print(
    "Retained rows:",
    int(
        mutation_exclusion_audit_df["ROW_STATUS"]
        .eq("Retained")
        .sum()
    )
)
print(
    "Excluded rows:",
    int(
        mutation_exclusion_audit_df["ROW_STATUS"]
        .eq("Excluded")
        .sum()
    )
)
print(
    "Overall exclusion rate:",
    round(overall_exclusion_rate, 2),
    "%"
)


# ---------------------------------------------------------
# 1. Exclusion by Variant_Classification
# ---------------------------------------------------------

variant_exclusion_summary = (
    mutation_exclusion_audit_df
    .groupby("Variant_Classification")
    .agg(
        TOTAL_ROWS=("ROW_STATUS", "size"),
        RETAINED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Retained").sum())
        ),
        EXCLUDED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Excluded").sum())
        )
    )
    .reset_index()
)

variant_exclusion_summary["EXCLUSION_RATE_PERCENT"] = (
    variant_exclusion_summary["EXCLUDED_ROWS"]
    / variant_exclusion_summary["TOTAL_ROWS"]
    * 100
).round(2)

variant_exclusion_summary[
    "DIFFERENCE_FROM_OVERALL_PERCENT"
] = (
    variant_exclusion_summary["EXCLUSION_RATE_PERCENT"]
    - overall_exclusion_rate
).round(2)

variant_exclusion_summary = (
    variant_exclusion_summary
    .sort_values(
        ["EXCLUDED_ROWS", "EXCLUSION_RATE_PERCENT"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

print("\nExclusion by Variant_Classification:")
display(variant_exclusion_summary)


# ---------------------------------------------------------
# 2. Binary-stage exclusion audit
# ---------------------------------------------------------

binary_stage_summary = (
    mutation_exclusion_audit_df
    .groupby("STAGE_BINARY")
    .agg(
        TOTAL_ROWS=("ROW_STATUS", "size"),
        RETAINED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Retained").sum())
        ),
        EXCLUDED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Excluded").sum())
        )
    )
    .reset_index()
)

binary_stage_summary["EXCLUSION_RATE_PERCENT"] = (
    binary_stage_summary["EXCLUDED_ROWS"]
    / binary_stage_summary["TOTAL_ROWS"]
    * 100
).round(2)

print("\nExclusion by Early/Advanced stage:")
display(binary_stage_summary)


# ---------------------------------------------------------
# 3. Four-class stage exclusion audit
# ---------------------------------------------------------

four_class_stage_summary = (
    mutation_exclusion_audit_df
    .groupby("STAGE_LABEL_4CLASS")
    .agg(
        TOTAL_ROWS=("ROW_STATUS", "size"),
        RETAINED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Retained").sum())
        ),
        EXCLUDED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Excluded").sum())
        )
    )
    .reset_index()
)

four_class_stage_summary["EXCLUSION_RATE_PERCENT"] = (
    four_class_stage_summary["EXCLUDED_ROWS"]
    / four_class_stage_summary["TOTAL_ROWS"]
    * 100
).round(2)

print("\nExclusion by Stage I/II/III/IV:")
display(four_class_stage_summary)


# ---------------------------------------------------------
# 4. Cohort-level exclusion audit
# ---------------------------------------------------------

cohort_exclusion_summary = (
    mutation_exclusion_audit_df
    .groupby("COHORT")
    .agg(
        TOTAL_ROWS=("ROW_STATUS", "size"),
        RETAINED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Retained").sum())
        ),
        EXCLUDED_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Excluded").sum())
        )
    )
    .reset_index()
)

cohort_exclusion_summary["EXCLUSION_RATE_PERCENT"] = (
    cohort_exclusion_summary["EXCLUDED_ROWS"]
    / cohort_exclusion_summary["TOTAL_ROWS"]
    * 100
).round(2)

cohort_exclusion_summary = (
    cohort_exclusion_summary
    .sort_values(
        "EXCLUSION_RATE_PERCENT",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nExclusion by cohort:")
display(cohort_exclusion_summary)

Total mutation rows: 15142
Retained rows: 12689
Excluded rows: 2453
Overall exclusion rate: 16.2 %

Exclusion by Variant_Classification:


,Variant_Classification,TOTAL_ROWS,RETAINED_ROWS,EXCLUDED_ROWS,EXCLUSION_RATE_PERCENT,DIFFERENCE_FROM_OVERALL_PERCENT
0,Missense_Mutation,9445,8652,793,8.40,-7.80
1,Splice_Site,752,5,747,99.34,83.14
2,Intron,296,0,296,100.00,83.80
3,Nonsense_Mutation,1624,1453,171,10.53,-5.67
4,Silent,773,652,121,15.65,-0.55
5,3'UTR,100,0,100,100.00,83.80
6,Splice_Region,107,36,71,66.36,50.16
7,5'UTR,31,0,31,100.00,83.80
8,3'Flank,28,0,28,100.00,83.80
9,Frame_Shift_Del,927,901,26,2.80,-13.40



Exclusion by Early/Advanced stage:


,STAGE_BINARY,TOTAL_ROWS,RETAINED_ROWS,EXCLUDED_ROWS,EXCLUSION_RATE_PERCENT
0,Advanced,5425,4594,831,15.32
1,Early,9717,8095,1622,16.69



Exclusion by Stage I/II/III/IV:


,STAGE_LABEL_4CLASS,TOTAL_ROWS,RETAINED_ROWS,EXCLUDED_ROWS,EXCLUSION_RATE_PERCENT
0,Stage I,7053,5859,1194,16.93
1,Stage II,2664,2236,428,16.07
2,Stage III,2383,1926,457,19.18
3,Stage IV,3042,2668,374,12.29



Exclusion by cohort:


,COHORT,TOTAL_ROWS,RETAINED_ROWS,EXCLUDED_ROWS,EXCLUSION_RATE_PERCENT
0,ONCOSG_2020,1650,1102,548,33.21
1,TCGA_Atlas_2018,4711,3912,799,16.96
2,MSKCC_2020,1768,1517,251,14.20
3,MSK_NPJPO_2021,1293,1115,178,13.77
4,NCI_2022,370,320,50,13.51
5,MSKCC_2023,2454,2131,323,13.16
6,LUNG_MSK_2017,2896,2592,304,10.50


In [6]:
# Step 6 — Check patient-level impact of excluded mutation rows

patient_exclusion_summary = (
    mutation_exclusion_audit_df
    .groupby(
        [
            "COHORT",
            "PATIENT_ID",
            "STAGE_BINARY",
            "STAGE_LABEL_4CLASS"
        ],
        dropna=False
    )
    .agg(
        TOTAL_MUTATION_ROWS=("ROW_STATUS", "size"),

        RETAINED_MUTATION_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Retained").sum())
        ),

        EXCLUDED_MUTATION_ROWS=(
            "ROW_STATUS",
            lambda x: int((x == "Excluded").sum())
        )
    )
    .reset_index()
)

patient_exclusion_summary["RETAINED_FRACTION"] = (
    patient_exclusion_summary["RETAINED_MUTATION_ROWS"]
    / patient_exclusion_summary["TOTAL_MUTATION_ROWS"]
)

patient_exclusion_summary["ALL_MUTATIONS_EXCLUDED"] = (
    patient_exclusion_summary["RETAINED_MUTATION_ROWS"] == 0
).astype(int)


print("Total unique patients:", len(patient_exclusion_summary))

print(
    "Patients with at least one retained mutation:",
    int(
        (
            patient_exclusion_summary["RETAINED_MUTATION_ROWS"] > 0
        ).sum()
    )
)

print(
    "Patients with all mutations excluded:",
    int(
        patient_exclusion_summary["ALL_MUTATIONS_EXCLUDED"].sum()
    )
)

print(
    "Overall median retained fraction:",
    round(
        patient_exclusion_summary["RETAINED_FRACTION"].median(),
        4
    )
)


# Patient-level audit by binary stage
patient_stage_audit = (
    patient_exclusion_summary
    .groupby("STAGE_BINARY")
    .agg(
        PATIENTS=("PATIENT_ID", "size"),

        ALL_EXCLUDED_PATIENTS=(
            "ALL_MUTATIONS_EXCLUDED",
            "sum"
        ),

        MEAN_RETAINED_FRACTION=(
            "RETAINED_FRACTION",
            "mean"
        ),

        MEDIAN_RETAINED_FRACTION=(
            "RETAINED_FRACTION",
            "median"
        )
    )
    .reset_index()
)

patient_stage_audit["ALL_EXCLUDED_PERCENT"] = (
    patient_stage_audit["ALL_EXCLUDED_PATIENTS"]
    / patient_stage_audit["PATIENTS"]
    * 100
).round(2)

patient_stage_audit["MEAN_RETAINED_FRACTION"] = (
    patient_stage_audit["MEAN_RETAINED_FRACTION"].round(4)
)

patient_stage_audit["MEDIAN_RETAINED_FRACTION"] = (
    patient_stage_audit["MEDIAN_RETAINED_FRACTION"].round(4)
)


# Patient-level audit by cohort
patient_cohort_audit = (
    patient_exclusion_summary
    .groupby("COHORT")
    .agg(
        PATIENTS=("PATIENT_ID", "size"),

        ALL_EXCLUDED_PATIENTS=(
            "ALL_MUTATIONS_EXCLUDED",
            "sum"
        ),

        MEAN_RETAINED_FRACTION=(
            "RETAINED_FRACTION",
            "mean"
        ),

        MEDIAN_RETAINED_FRACTION=(
            "RETAINED_FRACTION",
            "median"
        )
    )
    .reset_index()
)

patient_cohort_audit["ALL_EXCLUDED_PERCENT"] = (
    patient_cohort_audit["ALL_EXCLUDED_PATIENTS"]
    / patient_cohort_audit["PATIENTS"]
    * 100
).round(2)

patient_cohort_audit["MEAN_RETAINED_FRACTION"] = (
    patient_cohort_audit["MEAN_RETAINED_FRACTION"].round(4)
)

patient_cohort_audit["MEDIAN_RETAINED_FRACTION"] = (
    patient_cohort_audit["MEDIAN_RETAINED_FRACTION"].round(4)
)


print("\nPatient-level exclusion by binary stage:")
display(patient_stage_audit)

print("\nPatient-level exclusion by cohort:")
display(patient_cohort_audit)

Total unique patients: 3491
Patients with at least one retained mutation: 3382
Patients with all mutations excluded: 109
Overall median retained fraction: 1.0

Patient-level exclusion by binary stage:


,STAGE_BINARY,PATIENTS,ALL_EXCLUDED_PATIENTS,MEAN_RETAINED_FRACTION,MEDIAN_RETAINED_FRACTION,ALL_EXCLUDED_PERCENT
0,Advanced,1322,36,0.8620,1.0,2.72
1,Early,2169,73,0.8488,1.0,3.37



Patient-level exclusion by cohort:


,COHORT,PATIENTS,ALL_EXCLUDED_PATIENTS,MEAN_RETAINED_FRACTION,MEDIAN_RETAINED_FRACTION,ALL_EXCLUDED_PERCENT
0,LUNG_MSK_2017,802,23,0.8849,1.0000,2.87
1,MSKCC_2020,586,31,0.8494,1.0000,5.29
2,MSKCC_2023,765,20,0.8760,1.0000,2.61
3,MSK_NPJPO_2021,408,12,0.8699,1.0000,2.94
4,NCI_2022,158,7,0.8641,1.0000,4.43
5,ONCOSG_2020,279,11,0.7050,0.6667,3.94
6,TCGA_Atlas_2018,493,5,0.8417,0.8571,1.01


In [7]:
# Step 7 — Encode Variant_Classification into severity tiers

HIGH_IMPACT_VARIANTS = {
    "Nonsense_Mutation",
    "Nonstop_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "Splice_Site",
    "Translation_Start_Site"
}

MODERATE_IMPACT_VARIANTS = {
    "Missense_Mutation",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Region"
}

LOW_IMPACT_VARIANTS = {
    "Silent"
}


def map_mutation_severity(variant_classification):
    if pd.isna(variant_classification):
        return "Unknown", 0

    variant = str(variant_classification).strip()

    if variant in HIGH_IMPACT_VARIANTS:
        return "High", 3

    if variant in MODERATE_IMPACT_VARIANTS:
        return "Moderate", 2

    if variant in LOW_IMPACT_VARIANTS:
        return "Low", 1

    return "Other", 0


severity_results = valid_mutation_df[
    "Variant_Classification"
].apply(map_mutation_severity)

valid_mutation_df["MUTATION_SEVERITY"] = (
    severity_results.apply(lambda result: result[0])
)

valid_mutation_df["MUTATION_SEVERITY_SCORE"] = (
    severity_results.apply(lambda result: result[1])
)

# One-hot columns for ablation-ready modelling
severity_one_hot = pd.get_dummies(
    valid_mutation_df["MUTATION_SEVERITY"],
    prefix="severity",
    dtype=int
)

expected_severity_columns = [
    "severity_High",
    "severity_Moderate",
    "severity_Low",
    "severity_Other",
    "severity_Unknown"
]

for column in expected_severity_columns:
    if column not in severity_one_hot.columns:
        severity_one_hot[column] = 0

valid_mutation_df = pd.concat(
    [
        valid_mutation_df,
        severity_one_hot[expected_severity_columns]
    ],
    axis=1
)

print("Mutation severity distribution:")
print(
    valid_mutation_df["MUTATION_SEVERITY"]
    .value_counts()
    .to_string()
)

print("\nSeverity-score distribution:")
print(
    valid_mutation_df["MUTATION_SEVERITY_SCORE"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\nVariant classification to severity mapping:")
display(
    valid_mutation_df[
        [
            "Variant_Classification",
            "MUTATION_SEVERITY",
            "MUTATION_SEVERITY_SCORE"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "MUTATION_SEVERITY_SCORE",
            "Variant_Classification"
        ],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

Mutation severity distribution:
MUTATION_SEVERITY
Moderate    9394
High        2643
Low          652

Severity-score distribution:
MUTATION_SEVERITY_SCORE
1     652
2    9394
3    2643

Variant classification to severity mapping:


,Variant_Classification,MUTATION_SEVERITY,MUTATION_SEVERITY_SCORE
0,Frame_Shift_Del,High,3
1,Frame_Shift_Ins,High,3
2,Nonsense_Mutation,High,3
3,Nonstop_Mutation,High,3
4,Splice_Site,High,3
5,Translation_Start_Site,High,3
6,In_Frame_Del,Moderate,2
7,In_Frame_Ins,Moderate,2
8,Missense_Mutation,Moderate,2
9,Splice_Region,Moderate,2


In [8]:
# Step 8 — Align mutation rows with the selected sample used in final dataset

selected_patient_samples = (
    clinical_df[
        [
            "COHORT",
            "PATIENT_ID",
            "SAMPLE_ID"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print(
    "Selected patient records:",
    len(selected_patient_samples)
)

print(
    "Duplicate COHORT + PATIENT_ID:",
    selected_patient_samples.duplicated(
        subset=["COHORT", "PATIENT_ID"]
    ).sum()
)

# Keep mutation rows belonging to the selected patient sample
aligned_mutation_df = valid_mutation_df.merge(
    selected_patient_samples,
    on=[
        "COHORT",
        "PATIENT_ID",
        "SAMPLE_ID"
    ],
    how="inner",
    validate="many_to_one"
)

aligned_patient_count = (
    aligned_mutation_df[
        ["COHORT", "PATIENT_ID"]
    ]
    .drop_duplicates()
    .shape[0]
)

removed_nonselected_sample_rows = (
    len(valid_mutation_df)
    - len(aligned_mutation_df)
)

print("\nValid mutation rows before alignment:", len(valid_mutation_df))
print("Rows after selected-sample alignment:", len(aligned_mutation_df))
print(
    "Rows removed from non-selected samples:",
    removed_nonselected_sample_rows
)

print(
    "Patients represented after alignment:",
    aligned_patient_count
)

print(
    "Expected final patients:",
    len(selected_patient_samples)
)

print(
    "Patients missing mutation rows:",
    len(selected_patient_samples) - aligned_patient_count
)

print("\nAligned mutation counts by cohort:")
display(
    aligned_mutation_df
    .groupby("COHORT")
    .agg(
        PATIENTS=("PATIENT_ID", "nunique"),
        MUTATION_ROWS=("PATIENT_ID", "size")
    )
    .reset_index()
)

Selected patient records: 3382
Duplicate COHORT + PATIENT_ID: 0

Valid mutation rows before alignment: 12689
Rows after selected-sample alignment: 12580
Rows removed from non-selected samples: 109
Patients represented after alignment: 3382
Expected final patients: 3382
Patients missing mutation rows: 0

Aligned mutation counts by cohort:


,COHORT,PATIENTS,MUTATION_ROWS
0,LUNG_MSK_2017,779,2483
1,MSKCC_2020,555,1517
2,MSKCC_2023,745,2131
3,MSK_NPJPO_2021,396,1115
4,NCI_2022,151,320
5,ONCOSG_2020,268,1102
6,TCGA_Atlas_2018,488,3912


In [9]:
# Step 9 — Aggregate mutation-severity features per patient

patient_severity_features = (
    aligned_mutation_df
    .groupby(
        ["COHORT", "SAMPLE_ID", "PATIENT_ID"],
        as_index=False
    )
    .agg(
        TOTAL_VALID_MUTATIONS=(
            "MUTATION_SEVERITY_SCORE",
            "size"
        ),

        UNIQUE_MUTATED_GENES=(
            "GENE_SYMBOL",
            "nunique"
        ),

        MAX_SEVERITY_SCORE=(
            "MUTATION_SEVERITY_SCORE",
            "max"
        ),

        MEAN_SEVERITY_SCORE=(
            "MUTATION_SEVERITY_SCORE",
            "mean"
        ),

        HIGH_SEVERITY_COUNT=(
            "MUTATION_SEVERITY",
            lambda values: int(
                (values == "High").sum()
            )
        ),

        MODERATE_SEVERITY_COUNT=(
            "MUTATION_SEVERITY",
            lambda values: int(
                (values == "Moderate").sum()
            )
        ),

        LOW_SEVERITY_COUNT=(
            "MUTATION_SEVERITY",
            lambda values: int(
                (values == "Low").sum()
            )
        )
    )
)

patient_severity_features["MEAN_SEVERITY_SCORE"] = (
    patient_severity_features["MEAN_SEVERITY_SCORE"]
    .round(4)
)

duplicate_count = patient_severity_features.duplicated(
    subset=["COHORT", "PATIENT_ID"]
).sum()

print(
    "Patient-level severity shape:",
    patient_severity_features.shape
)

print(
    "Unique patients:",
    patient_severity_features[
        ["COHORT", "PATIENT_ID"]
    ].drop_duplicates().shape[0]
)

print(
    "Duplicate COHORT + PATIENT_ID:",
    duplicate_count
)

print("\nSeverity-feature summary:")
display(
    patient_severity_features[
        [
            "TOTAL_VALID_MUTATIONS",
            "UNIQUE_MUTATED_GENES",
            "MAX_SEVERITY_SCORE",
            "MEAN_SEVERITY_SCORE",
            "HIGH_SEVERITY_COUNT",
            "MODERATE_SEVERITY_COUNT",
            "LOW_SEVERITY_COUNT"
        ]
    ].describe().round(3)
)

print("\nSample patient-level severity features:")
display(patient_severity_features.head(10))

Patient-level severity shape: (3382, 10)
Unique patients: 3382
Duplicate COHORT + PATIENT_ID: 0

Severity-feature summary:


,TOTAL_VALID_MUTATIONS,UNIQUE_MUTATED_GENES,MAX_SEVERITY_SCORE,MEAN_SEVERITY_SCORE,HIGH_SEVERITY_COUNT,MODERATE_SEVERITY_COUNT,LOW_SEVERITY_COUNT
count,3382.000,3382.000,3382.000,3382.000,3382.000,3382.000,3382.000
mean,3.720,3.305,2.506,2.187,0.774,2.753,0.193
std,3.598,2.717,0.502,0.280,0.989,2.633,0.735
min,1.000,1.000,1.000,1.000,0.000,0.000,0.000
25%,2.000,2.000,2.000,2.000,0.000,1.000,0.000
50%,3.000,3.000,3.000,2.000,1.000,2.000,0.000
75%,4.000,4.000,3.000,2.333,1.000,3.000,0.000
max,37.000,24.000,3.000,3.000,8.000,28.000,9.000



Sample patient-level severity features:


,COHORT,SAMPLE_ID,PATIENT_ID,TOTAL_VALID_MUTATIONS,UNIQUE_MUTATED_GENES,MAX_SEVERITY_SCORE,MEAN_SEVERITY_SCORE,HIGH_SEVERITY_COUNT,MODERATE_SEVERITY_COUNT,LOW_SEVERITY_COUNT
0,LUNG_MSK_2017,P-0000036-T01-IM3,P-0000036,3,3,3,2.3333,1,2,0
1,LUNG_MSK_2017,P-0000110-T01-IM3,P-0000110,2,2,3,2.5000,1,1,0
2,LUNG_MSK_2017,P-0000133-T01-IM3,P-0000133,3,3,2,2.0000,0,3,0
3,LUNG_MSK_2017,P-0000149-T01-IM3,P-0000149,3,3,2,2.0000,0,3,0
4,LUNG_MSK_2017,P-0000163-T02-IM3,P-0000163,3,3,2,2.0000,0,3,0
5,LUNG_MSK_2017,P-0000165-T01-IM3,P-0000165,4,4,2,2.0000,0,4,0
6,LUNG_MSK_2017,P-0000205-T01-IM3,P-0000205,1,1,2,2.0000,0,1,0
7,LUNG_MSK_2017,P-0000219-T01-IM3,P-0000219,3,3,2,2.0000,0,3,0
8,LUNG_MSK_2017,P-0000231-T01-IM3,P-0000231,1,1,2,2.0000,0,1,0
9,LUNG_MSK_2017,P-0000233-T01-IM3,P-0000233,1,1,2,2.0000,0,1,0


In [10]:
# Step 10 — Create aligned patient-level mutation feature lists

patient_mutation_lists = (
    aligned_mutation_df
    .groupby(
        ["COHORT", "SAMPLE_ID", "PATIENT_ID"],
        as_index=False
    )
    .agg(
        MUTATED_GENE_LIST=(
            "GENE_SYMBOL",
            list
        ),

        REFERENCE_DNA_WINDOW_LIST=(
            "REFERENCE_DNA_WINDOW",
            list
        ),

        MUTATED_DNA_WINDOW_LIST=(
            "MUTATED_DNA_WINDOW",
            list
        ),

        MUTATION_POSITION_MASK_LIST=(
            "MUTATION_POSITION_MASK",
            list
        ),

        VARIANT_CLASSIFICATION_LIST=(
            "Variant_Classification",
            list
        ),

        MUTATION_SEVERITY_LIST=(
            "MUTATION_SEVERITY",
            list
        ),

        MUTATION_SEVERITY_SCORE_LIST=(
            "MUTATION_SEVERITY_SCORE",
            list
        ),

        MASK_EVENT_TYPE_LIST=(
            "MASK_EVENT_TYPE",
            list
        ),

        MUTATION_INDEX_LIST=(
            "MUTATION_INDEX",
            list
        )
    )
)


# Number of mutation windows per patient
patient_mutation_lists["MUTATION_WINDOW_COUNT"] = (
    patient_mutation_lists["MUTATED_GENE_LIST"]
    .apply(len)
)


# Check that all per-mutation lists remain aligned
list_columns = [
    "MUTATED_GENE_LIST",
    "REFERENCE_DNA_WINDOW_LIST",
    "MUTATED_DNA_WINDOW_LIST",
    "MUTATION_POSITION_MASK_LIST",
    "VARIANT_CLASSIFICATION_LIST",
    "MUTATION_SEVERITY_LIST",
    "MUTATION_SEVERITY_SCORE_LIST",
    "MASK_EVENT_TYPE_LIST",
    "MUTATION_INDEX_LIST"
]


def check_list_alignment(row):
    lengths = [
        len(row[column])
        for column in list_columns
    ]

    return len(set(lengths)) == 1


patient_mutation_lists["LISTS_ALIGNED"] = (
    patient_mutation_lists.apply(
        check_list_alignment,
        axis=1
    )
)


# Validate DNA-window and mask lengths
patient_mutation_lists["REFERENCE_WINDOWS_ALL_101"] = (
    patient_mutation_lists[
        "REFERENCE_DNA_WINDOW_LIST"
    ].apply(
        lambda windows: all(
            isinstance(window, str)
            and len(window) == 101
            for window in windows
        )
    )
)

patient_mutation_lists["MUTATED_WINDOWS_ALL_101"] = (
    patient_mutation_lists[
        "MUTATED_DNA_WINDOW_LIST"
    ].apply(
        lambda windows: all(
            isinstance(window, str)
            and len(window) == 101
            for window in windows
        )
    )
)

patient_mutation_lists["MASKS_ALL_101"] = (
    patient_mutation_lists[
        "MUTATION_POSITION_MASK_LIST"
    ].apply(
        lambda masks: all(
            len(mask) == 101
            and set(mask).issubset({0, 1})
            and sum(mask) >= 1
            for mask in masks
        )
    )
)


duplicate_patients = patient_mutation_lists.duplicated(
    subset=["COHORT", "PATIENT_ID"]
).sum()


print("Patient mutation-list shape:", patient_mutation_lists.shape)
print("Unique patients:", len(patient_mutation_lists))
print("Duplicate COHORT + PATIENT_ID:", duplicate_patients)

print(
    "Patients with aligned lists:",
    int(patient_mutation_lists["LISTS_ALIGNED"].sum()),
    "/",
    len(patient_mutation_lists)
)

print(
    "Patients with valid reference windows:",
    int(
        patient_mutation_lists[
            "REFERENCE_WINDOWS_ALL_101"
        ].sum()
    )
)

print(
    "Patients with valid mutated windows:",
    int(
        patient_mutation_lists[
            "MUTATED_WINDOWS_ALL_101"
        ].sum()
    )
)

print(
    "Patients with valid masks:",
    int(
        patient_mutation_lists[
            "MASKS_ALL_101"
        ].sum()
    )
)

print("\nMutation-window count summary:")
display(
    patient_mutation_lists[
        "MUTATION_WINDOW_COUNT"
    ].describe()
)

print("\nSample patient-level mutation lists:")
display(
    patient_mutation_lists[
        [
            "COHORT",
            "SAMPLE_ID",
            "PATIENT_ID",
            "MUTATION_WINDOW_COUNT",
            "MUTATED_GENE_LIST",
            "VARIANT_CLASSIFICATION_LIST",
            "MUTATION_SEVERITY_SCORE_LIST",
            "MASK_EVENT_TYPE_LIST",
            "LISTS_ALIGNED"
        ]
    ].head()
)

Patient mutation-list shape: (3382, 17)
Unique patients: 3382
Duplicate COHORT + PATIENT_ID: 0
Patients with aligned lists: 3382 / 3382
Patients with valid reference windows: 3382
Patients with valid mutated windows: 3382
Patients with valid masks: 3382

Mutation-window count summary:


count    3382.000000
mean        3.719692
std         3.598377
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        37.000000
Name: MUTATION_WINDOW_COUNT, dtype: float64


Sample patient-level mutation lists:


,COHORT,SAMPLE_ID,PATIENT_ID,MUTATION_WINDOW_COUNT,MUTATED_GENE_LIST,VARIANT_CLASSIFICATION_LIST,MUTATION_SEVERITY_SCORE_LIST,MASK_EVENT_TYPE_LIST,LISTS_ALIGNED
0,LUNG_MSK_2017,P-0000036-T01-IM3,P-0000036,3,"[TP53, ERBB2, FBXW7]","[Nonsense_Mutation, Missense_Mutation, In_Fram...","[3, 2, 2]","[SNV, SNV, DELETION]",True
1,LUNG_MSK_2017,P-0000110-T01-IM3,P-0000110,2,"[KEAP1, TP53]","[Missense_Mutation, Frame_Shift_Del]","[2, 3]","[SNV, DELETION]",True
2,LUNG_MSK_2017,P-0000133-T01-IM3,P-0000133,3,"[ATM, PIK3CA, TP53]","[Missense_Mutation, Missense_Mutation, Missens...","[2, 2, 2]","[SNV, SNV, SNV]",True
3,LUNG_MSK_2017,P-0000149-T01-IM3,P-0000149,3,"[SETD2, TSC1, ERBB2]","[Missense_Mutation, Missense_Mutation, In_Fram...","[2, 2, 2]","[SNV, SNV, INSERTION]",True
4,LUNG_MSK_2017,P-0000163-T02-IM3,P-0000163,3,"[TP53, SMARCA4, ERBB2]","[Missense_Mutation, Missense_Mutation, In_Fram...","[2, 2, 2]","[SNV, SNV, INSERTION]",True


In [11]:
# Step 11 — Create patient-level 96-gene multi-hot features

GENE_PANEL_PATH = (
    PROJECT_DIR
    / "processed"
    / "LUAD_driver_genes.csv"
)

gene_panel_df = pd.read_csv(
    GENE_PANEL_PATH,
    low_memory=False
)

# Identify gene-symbol column
if "SYMBOL" in gene_panel_df.columns:
    gene_column = "SYMBOL"
elif "GENE_SYMBOL" in gene_panel_df.columns:
    gene_column = "GENE_SYMBOL"
else:
    raise ValueError(
        "Gene-symbol column not found in LUAD driver-gene file."
    )

selected_genes = (
    gene_panel_df[gene_column]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

selected_gene_set = set(selected_genes)

print("Selected LUAD genes:", len(selected_genes))


def create_gene_multihot(gene_list):
    patient_genes = set(
        str(gene).strip()
        for gene in gene_list
    )

    return [
        int(gene in patient_genes)
        for gene in selected_genes
    ]


patient_gene_multihot = patient_mutation_lists[
    [
        "COHORT",
        "SAMPLE_ID",
        "PATIENT_ID",
        "MUTATED_GENE_LIST"
    ]
].copy()

patient_gene_multihot["GENE_MULTIHOT_VECTOR"] = (
    patient_gene_multihot["MUTATED_GENE_LIST"]
    .apply(create_gene_multihot)
)

# Create individual columns for ablation and classical ML models
gene_feature_columns = [
    f"GENE_{gene}"
    for gene in selected_genes
]

gene_feature_df = pd.DataFrame(
    patient_gene_multihot[
        "GENE_MULTIHOT_VECTOR"
    ].tolist(),
    columns=gene_feature_columns,
    index=patient_gene_multihot.index
)

patient_gene_multihot = pd.concat(
    [
        patient_gene_multihot,
        gene_feature_df
    ],
    axis=1
)

# Validation
patient_gene_multihot["MULTIHOT_ACTIVE_GENES"] = (
    gene_feature_df.sum(axis=1)
)

patient_gene_multihot["GENE_LIST_UNIQUE_COUNT"] = (
    patient_gene_multihot["MUTATED_GENE_LIST"]
    .apply(lambda genes: len(set(genes)))
)

count_mismatches = (
    patient_gene_multihot["MULTIHOT_ACTIVE_GENES"]
    != patient_gene_multihot["GENE_LIST_UNIQUE_COUNT"]
).sum()

genes_outside_panel = sorted(
    {
        gene
        for gene_list in patient_gene_multihot["MUTATED_GENE_LIST"]
        for gene in gene_list
        if gene not in selected_gene_set
    }
)

print("Patient rows:", len(patient_gene_multihot))
print("Gene feature columns:", len(gene_feature_columns))
print("Gene-count mismatches:", count_mismatches)
print("Genes outside selected panel:", genes_outside_panel)

print("\nActive gene-count summary:")
display(
    patient_gene_multihot[
        "MULTIHOT_ACTIVE_GENES"
    ].describe()
)

print("\nSample multi-hot records:")
display(
    patient_gene_multihot[
        [
            "COHORT",
            "PATIENT_ID",
            "MUTATED_GENE_LIST",
            "MULTIHOT_ACTIVE_GENES",
            "GENE_MULTIHOT_VECTOR"
        ]
    ].head()
)

Selected LUAD genes: 96
Patient rows: 3382
Gene feature columns: 96
Gene-count mismatches: 0
Genes outside selected panel: []

Active gene-count summary:


count    3382.000000
mean        3.305441
std         2.716817
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        24.000000
Name: MULTIHOT_ACTIVE_GENES, dtype: float64


Sample multi-hot records:


,COHORT,PATIENT_ID,MUTATED_GENE_LIST,MULTIHOT_ACTIVE_GENES,GENE_MULTIHOT_VECTOR
0,LUNG_MSK_2017,P-0000036,"[TP53, ERBB2, FBXW7]",3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,LUNG_MSK_2017,P-0000110,"[KEAP1, TP53]",2,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,LUNG_MSK_2017,P-0000133,"[ATM, PIK3CA, TP53]",3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, ..."
3,LUNG_MSK_2017,P-0000149,"[SETD2, TSC1, ERBB2]",3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,LUNG_MSK_2017,P-0000163,"[TP53, SMARCA4, ERBB2]",3,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [12]:
# Step 12 — Merge DNA, masks, severity, genes and clinical features

FINAL_BASELINE_PATH = (
    PROJECT_DIR
    / "luad_processed"
    / "final_luad_minion_patient_dataset.csv"
)

baseline_patient_df = pd.read_csv(
    FINAL_BASELINE_PATH,
    low_memory=False
)

merge_keys = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID"
]

# Keep patient information and prediction labels
patient_metadata_df = baseline_patient_df[
    [
        "COHORT",
        "SAMPLE_ID",
        "PATIENT_ID",
        "SEX",
        "sex_encoded",
        "STAGE_LABEL_4CLASS",
        "STAGE_BINARY",
        "label_stage_4class",
        "label_stage_binary"
    ]
].copy()


# Severity features without repeating identifiers
severity_columns = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "TOTAL_VALID_MUTATIONS",
    "UNIQUE_MUTATED_GENES",
    "MAX_SEVERITY_SCORE",
    "MEAN_SEVERITY_SCORE",
    "HIGH_SEVERITY_COUNT",
    "MODERATE_SEVERITY_COUNT",
    "LOW_SEVERITY_COUNT"
]

severity_for_merge = patient_severity_features[
    severity_columns
].copy()


# Gene multi-hot features
gene_columns_for_merge = (
    merge_keys
    + ["GENE_MULTIHOT_VECTOR", "MULTIHOT_ACTIVE_GENES"]
    + gene_feature_columns
)

genes_for_merge = patient_gene_multihot[
    gene_columns_for_merge
].copy()


# Clinical features
# Smoking is retained for ablation, but excluded from the primary model
clinical_columns_for_merge = [
    "COHORT",
    "SAMPLE_ID",
    "PATIENT_ID",
    "age_numeric",
    "age_for_model",
    "age_unusable_flag",
    "age_outlier_flag",
    "smoking_detail",
    "smoking_binary",
    "smoking_unknown_flag",
    "smoking_pack_years_numeric",
    "pack_years_missing_flag"
]

clinical_for_final_merge = clinical_df[
    clinical_columns_for_merge
].copy()


# Sequential one-to-one merges
final_improved_patient_df = patient_metadata_df.merge(
    patient_mutation_lists,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

final_improved_patient_df = final_improved_patient_df.merge(
    severity_for_merge,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

final_improved_patient_df = final_improved_patient_df.merge(
    genes_for_merge,
    on=merge_keys,
    how="inner",
    validate="one_to_one"
)

final_improved_patient_df = final_improved_patient_df.merge(
    clinical_for_final_merge,
    on=merge_keys,
    how="left",
    validate="one_to_one"
)


# Final validation
duplicate_count = final_improved_patient_df.duplicated(
    subset=["COHORT", "PATIENT_ID"]
).sum()

missing_binary_labels = (
    final_improved_patient_df["label_stage_binary"]
    .isna()
    .sum()
)

missing_four_class_labels = (
    final_improved_patient_df["label_stage_4class"]
    .isna()
    .sum()
)

print("Final improved dataset shape:",
      final_improved_patient_df.shape)

print("Final patients:", len(final_improved_patient_df))
print("Duplicate COHORT + PATIENT_ID:", duplicate_count)

print(
    "Missing binary labels:",
    missing_binary_labels
)

print(
    "Missing four-class labels:",
    missing_four_class_labels
)

print(
    "Aligned mutation lists:",
    int(final_improved_patient_df["LISTS_ALIGNED"].sum()),
    "/",
    len(final_improved_patient_df)
)

print(
    "Valid reference windows:",
    int(
        final_improved_patient_df[
            "REFERENCE_WINDOWS_ALL_101"
        ].sum()
    )
)

print(
    "Valid mutated windows:",
    int(
        final_improved_patient_df[
            "MUTATED_WINDOWS_ALL_101"
        ].sum()
    )
)

print(
    "Valid position masks:",
    int(
        final_improved_patient_df[
            "MASKS_ALL_101"
        ].sum()
    )
)

print(
    "Usable age:",
    int(
        final_improved_patient_df[
            "age_for_model"
        ].notna().sum()
    )
)

print(
    "Known smoking:",
    int(
        final_improved_patient_df[
            "smoking_binary"
        ].isin(["Never", "Ever"]).sum()
    )
)

print(
    "Gene multi-hot columns:",
    len(gene_feature_columns)
)

Final improved dataset shape: (3382, 137)
Final patients: 3382
Duplicate COHORT + PATIENT_ID: 0
Missing binary labels: 0
Missing four-class labels: 0
Aligned mutation lists: 3382 / 3382
Valid reference windows: 3382
Valid mutated windows: 3382
Valid position masks: 3382
Usable age: 3349
Known smoking: 2709
Gene multi-hot columns: 96


In [13]:
# Step 13 — Check new features for possible stage-label leakage

from sklearn.metrics import roc_auc_score

TARGET_COLUMN = "label_stage_binary"

# Primary-model numeric features
numeric_feature_columns = [
    "sex_encoded",
    "age_for_model",
    "age_unusable_flag",
    "age_outlier_flag",
    "TOTAL_VALID_MUTATIONS",
    "UNIQUE_MUTATED_GENES",
    "MAX_SEVERITY_SCORE",
    "MEAN_SEVERITY_SCORE",
    "HIGH_SEVERITY_COUNT",
    "MODERATE_SEVERITY_COUNT",
    "LOW_SEVERITY_COUNT",
    "smoking_unknown_flag"
]

# Include 96 gene multi-hot columns in the audit
audit_numeric_columns = (
    numeric_feature_columns
    + gene_feature_columns
)

target_values = final_improved_patient_df[
    TARGET_COLUMN
].astype(int)

audit_rows = []

for column in audit_numeric_columns:

    feature_values = pd.to_numeric(
        final_improved_patient_df[column],
        errors="coerce"
    )

    valid_rows = (
        feature_values.notna()
        & target_values.notna()
    )

    if valid_rows.sum() == 0:
        continue

    unique_values = feature_values.loc[
        valid_rows
    ].nunique()

    correlation = feature_values.loc[
        valid_rows
    ].corr(
        target_values.loc[valid_rows]
    )

    # Calculate univariate AUC where the feature varies
    if unique_values > 1:
        try:
            auc_value = roc_auc_score(
                target_values.loc[valid_rows],
                feature_values.loc[valid_rows]
            )

            # Direction-independent AUC
            auc_strength = max(
                auc_value,
                1 - auc_value
            )

        except ValueError:
            auc_strength = np.nan
    else:
        auc_strength = np.nan

    absolute_correlation = (
        abs(correlation)
        if pd.notna(correlation)
        else np.nan
    )

    if (
        pd.notna(absolute_correlation)
        and absolute_correlation >= 0.70
    ):
        warning = "HIGH — possible leakage"

    elif (
        pd.notna(auc_strength)
        and auc_strength >= 0.80
    ):
        warning = "HIGH — inspect carefully"

    elif (
        pd.notna(absolute_correlation)
        and absolute_correlation >= 0.40
    ):
        warning = "MODERATE — review"

    elif (
        pd.notna(auc_strength)
        and auc_strength >= 0.70
    ):
        warning = "MODERATE — review"

    else:
        warning = "No strong leakage signal"

    audit_rows.append(
        {
            "FEATURE": column,
            "NON_MISSING": int(valid_rows.sum()),
            "UNIQUE_VALUES": int(unique_values),
            "CORRELATION_WITH_STAGE": correlation,
            "ABS_CORRELATION": absolute_correlation,
            "UNIVARIATE_AUC_STRENGTH": auc_strength,
            "WARNING": warning
        }
    )


feature_leakage_audit = pd.DataFrame(
    audit_rows
).sort_values(
    [
        "UNIVARIATE_AUC_STRENGTH",
        "ABS_CORRELATION"
    ],
    ascending=False
).reset_index(drop=True)

feature_leakage_audit[
    [
        "CORRELATION_WITH_STAGE",
        "ABS_CORRELATION",
        "UNIVARIATE_AUC_STRENGTH"
    ]
] = feature_leakage_audit[
    [
        "CORRELATION_WITH_STAGE",
        "ABS_CORRELATION",
        "UNIVARIATE_AUC_STRENGTH"
    ]
].round(4)


print("Top features by univariate stage association:")

display(
    feature_leakage_audit.head(20)
)


# ---------------------------------------------------------
# Category-wise Advanced-stage percentages
# ---------------------------------------------------------

def categorical_stage_summary(dataframe, column):
    summary = (
        dataframe
        .groupby(column, dropna=False)
        .agg(
            PATIENTS=(TARGET_COLUMN, "size"),
            ADVANCED_PATIENTS=(
                TARGET_COLUMN,
                "sum"
            ),
            ADVANCED_PERCENT=(
                TARGET_COLUMN,
                lambda x: round(x.mean() * 100, 2)
            )
        )
        .reset_index()
    )

    return summary


print("\nStage distribution by sex:")
display(
    categorical_stage_summary(
        final_improved_patient_df,
        "SEX"
    )
)

print("\nStage distribution by smoking category:")
display(
    categorical_stage_summary(
        final_improved_patient_df,
        "smoking_binary"
    )
)

print("\nStage distribution by cohort — audit only:")
display(
    categorical_stage_summary(
        final_improved_patient_df,
        "COHORT"
    )
)


# ---------------------------------------------------------
# Search column names for direct stage/TNM leakage fields
# ---------------------------------------------------------

leakage_keywords = [
    "STAGE",
    "TNM",
    "PATH_T",
    "PATH_N",
    "PATH_M",
    "TUMOR_SIZE",
    "LYMPH",
    "METAST"
]

possible_leakage_columns = [
    column
    for column in final_improved_patient_df.columns
    if any(
        keyword in column.upper()
        for keyword in leakage_keywords
    )
]

print("\nColumns containing stage/TNM-related keywords:")
print(possible_leakage_columns)

print(
    "\nNote: prediction-label columns are expected here, "
    "but they must never be included in model inputs."
)

c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_ba

Top features by univariate stage association:


c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\LAPTOP\Downloads\luad_reserch_code\.venv\lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,FEATURE,NON_MISSING,UNIQUE_VALUES,CORRELATION_WITH_STAGE,ABS_CORRELATION,UNIVARIATE_AUC_STRENGTH,WARNING
0,age_for_model,3349,795,-0.1397,0.1397,0.5767,No strong leakage signal
1,GENE_TP53,3382,2,0.1284,0.1284,0.5655,No strong leakage signal
2,smoking_unknown_flag,3382,2,-0.1570,0.1570,0.5646,No strong leakage signal
3,MEAN_SEVERITY_SCORE,3382,92,0.0552,0.0552,0.5342,No strong leakage signal
4,LOW_SEVERITY_COUNT,3382,10,-0.0894,0.0894,0.5320,No strong leakage signal
5,GENE_KEAP1,3382,2,0.0774,0.0774,0.5275,No strong leakage signal
6,GENE_CSMD3,3382,2,-0.0992,0.0992,0.5275,No strong leakage signal
7,GENE_KRAS,3382,2,-0.0376,0.0376,0.5181,No strong leakage signal
8,GENE_ANK2,3382,2,-0.0798,0.0798,0.5165,No strong leakage signal
9,GENE_LRP1B,3382,2,-0.0592,0.0592,0.5153,No strong leakage signal



Stage distribution by sex:


...


Stage distribution by smoking category:


,smoking_binary,PATIENTS,ADVANCED_PATIENTS,ADVANCED_PERCENT
0,Ever,1899,742,39.07
1,Never,810,391,48.27
2,Unknown,673,153,22.73



Stage distribution by cohort — audit only:


,COHORT,PATIENTS,ADVANCED_PATIENTS,ADVANCED_PERCENT
0,LUNG_MSK_2017,779,695,89.22
1,MSKCC_2020,555,58,10.45
2,MSKCC_2023,745,302,40.54
3,MSK_NPJPO_2021,396,0,0.00
4,NCI_2022,151,30,19.87
5,ONCOSG_2020,268,94,35.07
6,TCGA_Atlas_2018,488,107,21.93



Columns containing stage/TNM-related keywords:
['STAGE_LABEL_4CLASS', 'STAGE_BINARY', 'label_stage_4class', 'label_stage_binary']

Note: prediction-label columns are expected here, but they must never be included in model inputs.


In [14]:
# Step 14 — Save final improved patient dataset and TXT summary

import json

FINAL_DATASET_PATH = (
    IMPROVED_OUTPUT_DIR
    / "final_luad_improved_patient_dataset.csv"
)

FINAL_SUMMARY_PATH = (
    IMPROVED_OUTPUT_DIR
    / "final_luad_improved_patient_dataset_summary.txt"
)

dataset_to_save = final_improved_patient_df.copy()

# Convert list-based columns to valid JSON strings
json_list_columns = [
    "MUTATED_GENE_LIST",
    "REFERENCE_DNA_WINDOW_LIST",
    "MUTATED_DNA_WINDOW_LIST",
    "MUTATION_POSITION_MASK_LIST",
    "VARIANT_CLASSIFICATION_LIST",
    "MUTATION_SEVERITY_LIST",
    "MUTATION_SEVERITY_SCORE_LIST",
    "MASK_EVENT_TYPE_LIST",
    "MUTATION_INDEX_LIST",
    "GENE_MULTIHOT_VECTOR"
]

for column in json_list_columns:
    dataset_to_save[column] = (
        dataset_to_save[column]
        .apply(json.dumps)
    )

# Save CSV — overwrite existing file
dataset_to_save.to_csv(
    FINAL_DATASET_PATH,
    index=False,
    mode="w"
)

# Identify constant gene columns for documentation
constant_gene_columns = [
    column
    for column in gene_feature_columns
    if final_improved_patient_df[column].nunique() <= 1
]

summary_lines = [
    "FINAL LUAD IMPROVED PATIENT DATASET SUMMARY",
    "=" * 65,
    "",
    f"Total patients: {len(dataset_to_save)}",
    f"Total columns: {dataset_to_save.shape[1]}",
    f"Unique cohorts: {dataset_to_save['COHORT'].nunique()}",
    "",
    "DATA QUALITY",
    "-" * 65,
    f"Duplicate COHORT + PATIENT_ID: "
    f"{dataset_to_save.duplicated(['COHORT', 'PATIENT_ID']).sum()}",
    f"Missing binary labels: "
    f"{dataset_to_save['label_stage_binary'].isna().sum()}",
    f"Missing four-class labels: "
    f"{dataset_to_save['label_stage_4class'].isna().sum()}",
    f"Patients with valid aligned mutation lists: "
    f"{int(final_improved_patient_df['LISTS_ALIGNED'].sum())}",
    f"Patients with valid reference windows: "
    f"{int(final_improved_patient_df['REFERENCE_WINDOWS_ALL_101'].sum())}",
    f"Patients with valid mutated windows: "
    f"{int(final_improved_patient_df['MUTATED_WINDOWS_ALL_101'].sum())}",
    f"Patients with valid mutation masks: "
    f"{int(final_improved_patient_df['MASKS_ALL_101'].sum())}",
    "",
    "MUTATION FEATURES",
    "-" * 65,
    f"Total valid mutation windows: "
    f"{int(final_improved_patient_df['TOTAL_VALID_MUTATIONS'].sum())}",
    f"Mean mutation windows per patient: "
    f"{final_improved_patient_df['TOTAL_VALID_MUTATIONS'].mean():.3f}",
    f"Maximum mutation windows per patient: "
    f"{int(final_improved_patient_df['TOTAL_VALID_MUTATIONS'].max())}",
    f"LUAD gene panel size: {len(gene_feature_columns)}",
    f"Constant/all-zero gene columns: {constant_gene_columns}",
    "",
    "CLINICAL FEATURES",
    "-" * 65,
    f"Usable age records: "
    f"{final_improved_patient_df['age_for_model'].notna().sum()} "
    f"({final_improved_patient_df['age_for_model'].notna().mean() * 100:.2f}%)",
    f"Known smoking records: "
    f"{final_improved_patient_df['smoking_binary'].isin(['Never', 'Ever']).sum()} "
    f"({final_improved_patient_df['smoking_binary'].isin(['Never', 'Ever']).mean() * 100:.2f}%)",
    "Passive smoking included: No",
    "Pack-years used in primary model: No",
    "",
    "PRIMARY MODEL FEATURES",
    "-" * 65,
    "Reference DNA windows",
    "Mutated DNA windows",
    "Mutation-position masks",
    "Mutation category/severity",
    "96-gene representation",
    "Sex",
    "Age",
    "",
    "ABLATION-ONLY FEATURE",
    "-" * 65,
    "Smoking history is preserved but excluded from the primary model.",
    "It is completely unavailable in TCGA_Atlas_2018 and NCI_2022.",
    "",
    "LEAKAGE AUDIT",
    "-" * 65,
    "No non-target numeric feature showed a strong univariate leakage signal.",
    "COHORT must not be used as a model input because stage distributions",
    "differ substantially across cohorts.",
    "Stage-label columns must only be used as prediction targets."
]

with open(
    FINAL_SUMMARY_PATH,
    mode="w",
    encoding="utf-8"
) as summary_file:
    summary_file.write("\n".join(summary_lines))

# Reload verification
saved_final_df = pd.read_csv(
    FINAL_DATASET_PATH,
    low_memory=False
)

print("Final dataset saved:")
print(FINAL_DATASET_PATH)

print("\nTXT summary saved:")
print(FINAL_SUMMARY_PATH)

print("\nSaved shape:", saved_final_df.shape)
print(
    "Duplicate patients:",
    saved_final_df.duplicated(
        subset=["COHORT", "PATIENT_ID"]
    ).sum()
)
print("Constant gene columns:", constant_gene_columns)

Final dataset saved:
C:\Users\LAPTOP\Downloads\luad_reserch_code\luad_processeed_data\final_luad_improved_patient_dataset.csv

TXT summary saved:
C:\Users\LAPTOP\Downloads\luad_reserch_code\luad_processeed_data\final_luad_improved_patient_dataset_summary.txt

Saved shape: (3382, 137)
Duplicate patients: 0
Constant gene columns: ['GENE_CAPN5', 'GENE_CBLB', 'GENE_CRLF2', 'GENE_EPPK1', 'GENE_NOTCH1', 'GENE_ZFHX4', 'GENE_ZNF724']
